In [6]:
import pandas as pd

In [7]:
import os

print(os.getcwd())

C:\Users\Shrusti R\Downloads\olist_project\notebook


In [8]:
os.listdir()

['.ipynb_checkpoints',
 'Data Cleaning & Preparation.ipynb',
 'DAY_1_EXPLORATION.ipynb',
 'DAY_2_Data_understanding.ipynb']

In [14]:
import pandas as pd
import numpy as np
customers = pd.read_csv(r"C:\Users\Shrusti R\Downloads\olist_project\data\olist_customers_dataset.csv")
geolocation = pd.read_csv(r"C:\Users\Shrusti R\Downloads\olist_project\data\olist_geolocation_dataset.csv")
order_items = pd.read_csv(r"C:\Users\Shrusti R\Downloads\olist_project\data\olist_order_items_dataset.csv")
payments = pd.read_csv(r"C:\Users\Shrusti R\Downloads\olist_project\data\olist_order_payments_dataset.csv")
reviews = pd.read_csv(r"C:\Users\Shrusti R\Downloads\olist_project\data\olist_order_reviews_dataset.csv")
orders = pd.read_csv(r"C:\Users\Shrusti R\Downloads\olist_project\data\olist_orders_dataset.csv")
products = pd.read_csv(r"C:\Users\Shrusti R\Downloads\olist_project\data\olist_products_dataset.csv")
sellers = pd.read_csv(r"C:\Users\Shrusti R\Downloads\olist_project\data\olist_sellers_dataset.csv")
category_translation = pd.read_csv(r"C:\Users\Shrusti R\Downloads\olist_project\data\product_category_name_translation.csv")
print(orders)

                               order_id                       customer_id  \
0      e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1      53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2      47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3      949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4      ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
...                                 ...                               ...   
99436  9c5dedf39a927c1b2549525ed64a053c  39bd1228ee8140590ac3aca26f2dfe00   
99437  63943bddc261676b46f01ca7ac2f7bd8  1fca14ff2861355f6e5f14306ff977a7   
99438  83c1379a015df1e13d02aae0204711ab  1aa71eb042121263aafbe80c1b562c9c   
99439  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
99440  66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   

      order_status order_purchase_timestamp    order_approved_at  \
0      

In [15]:
#missing data
datasets = {
    "Orders": orders,
    "Customers": customers,
    "Reviews": reviews,
    "Order Items": order_items,
    "Products": products,
    "Payments": payments,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Orders: (99441, 8)
Customers: (99441, 5)
Reviews: (100000, 7)
Order Items: (112650, 7)
Products: (32951, 9)
Payments: (103886, 5)
Sellers: (3095, 4)
Geolocation: (1000163, 5)
Category Translation: (71, 2)


In [16]:
#order that was cancelled might naturally have no delivery date.
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.isnull().sum())


ORDERS
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

CUSTOMERS
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

REVIEWS
review_id                      0
order_id                       0
review_score                   0
review_comment_title       88285
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

ORDER ITEMS
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

PRODUCTS
product_id                      0
pro

In [17]:
#Orders — Missing delivery dates
orders.groupby('order_status')[
    ['order_approved_at',
     'order_delivered_carrier_date',
     'order_delivered_customer_date']
].apply(lambda x: x.isna().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [18]:
#Reviews — Lots of missing comments
review_comment_analysis = reviews['review_comment_message'].isna().value_counts()

print(review_comment_analysis)

review_comment_message
True     58247
False    41753
Name: count, dtype: int64


In [20]:
#Calculate the percentage
comment_percentage = reviews['has_comment'].mean() * 100

print(f"Reviews with written comments: {comment_percentage:.2f}%")
print(f"Reviews without written comments: {100 - comment_percentage:.2f}%")

Reviews with written comments: 41.73%
Reviews without written comments: 58.27%


In [21]:
#Find how much information is missing in those 610 products
missing_category = products[products['product_category_name'].isna()]

missing_category.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [22]:
#whether products with missing categories are actually being purchased.
missing_category_sales = order_items[
    order_items['product_id'].isin(missing_category['product_id'])
]

missing_category_sales.shape


(1603, 7)

In [23]:
#Calculate the revenue associated with products whose category is missing.
(
    missing_category_sales['price'] +
    missing_category_sales['freight_value']
).sum()

np.float64(207705.09000000003)

In [24]:
missing_category.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [25]:
missing_category_sales.shape

(1603, 7)

In [26]:
missing_category_sales['product_id'].nunique()

610

In [27]:
missing_category_sales['price'].sum()

np.float64(179535.28)

                    DATA
                      ↓
              Missing values
                      ↓
          Is missing data legitimate?
                      ↓
               Duplicate records
                      ↓
              Incorrect data types
                      ↓
               Invalid dates
                      ↓
          Broken table relationships
                      ↓
            Inconsistent values
                      ↓
              Clean datasets
                      ↓
             Validate the fix